# 第一部分：数据获取

这个 Notebook 负责准备项目目录，并从 `akshare` 拉取后续分析要用的股票、指数、宏观和财务数据。下载是否成功会写入 `download_log.txt`，之后如果某个接口失败，可以直接回头查日志。


In [1]:

from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != 'dshw-p01':
    PROJECT_ROOT = PROJECT_ROOT / 'dshw-p01'
os.makedirs(PROJECT_ROOT, exist_ok=True)

for path in [
    'data/stock', 'data/index', 'data/macro', 'data/finance',
    'data/clean', 'data/combined', 'output'
]:
    os.makedirs(PROJECT_ROOT / path, exist_ok=True)

PROJECT_ROOT


PosixPath('/Users/wonderlab/Desktop/欣欣作业/ds2026/dshw-p01')

In [2]:

import os
import time
from datetime import datetime
import pandas as pd
import numpy as np
import akshare as ak

# 清理本地代理变量，避免 akshare 请求被代理中断。
for key in list(os.environ):
    if 'proxy' in key.lower():
        os.environ.pop(key, None)

START_DATE = '20200101'
START_DATE_DASH = '2020-01-01'
END_DATE = datetime.today().strftime('%Y%m%d')
LOG_PATH = PROJECT_ROOT / 'download_log.txt'

STOCKS = [
    {'code': '000625', 'ak_symbol': 'sz000625', 'name': '长安汽车', 'industry': '汽车'},
    {'code': '601127', 'ak_symbol': 'sh601127', 'name': '赛力斯', 'industry': '汽车'},
    {'code': '600246', 'ak_symbol': 'sh600246', 'name': '万通发展', 'industry': '房地产'},
    {'code': '000560', 'ak_symbol': 'sz000560', 'name': '我爱我家', 'industry': '房地产'},
    {'code': '600519', 'ak_symbol': 'sh600519', 'name': '贵州茅台', 'industry': '白酒'},
    {'code': '000799', 'ak_symbol': 'sz000799', 'name': '酒鬼酒', 'industry': '白酒'},
    {'code': '600578', 'ak_symbol': 'sh600578', 'name': '京能电力', 'industry': '能源'},
    {'code': '601016', 'ak_symbol': 'sh601016', 'name': '节能风电', 'industry': '能源'},
    {'code': '000063', 'ak_symbol': 'sz000063', 'name': '中兴通讯', 'industry': '通讯'},
    {'code': '300136', 'ak_symbol': 'sz300136', 'name': '信维通信', 'industry': '通讯'},
]

INDEXES = [
    {'code': '000300', 'ak_symbol': 'sh000300', 'name': '沪深300'},
    {'code': '000001', 'ak_symbol': 'sh000001', 'name': '上证综指'},
]

def log(status, item, detail=''):
    line = f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {status:<7} {item:<18} {detail}"
    print(line)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def normalize_stock_df(df, item):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'volume', 'amount', 'turnover']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df['code'] = item['code']
    df['name'] = item['name']
    df['industry'] = item['industry']
    if 'amount' not in df.columns:
        df['amount'] = np.nan
    keep = ['date', 'code', 'name', 'industry', 'open', 'close', 'high', 'low', 'volume', 'amount']
    return df[keep].sort_values('date')

def normalize_index_df(df, item):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'volume']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df['code'] = item['code']
    df['name'] = item['name']
    df['amount'] = np.nan
    keep = ['date', 'code', 'name', 'open', 'close', 'high', 'low', 'volume', 'amount']
    return df[keep].sort_values('date')


## 1.1 自选 10 只股票

股票行情来自 `akshare.stock_zh_a_daily()`，这里使用后复权价格，避免分红送转造成的价格断点。样本包含 10 只 A 股，覆盖汽车、房地产、白酒、能源和通讯 5 个行业。


In [3]:

if LOG_PATH.exists():
    LOG_PATH.unlink()

for item in STOCKS:
    try:
        df = ak.stock_zh_a_daily(
            symbol=item['ak_symbol'],
            start_date=START_DATE,
            end_date=END_DATE,
            adjust='hfq'
        )
        if df.empty:
            raise ValueError('No data returned')
        out = normalize_stock_df(df, item)
        path = PROJECT_ROOT / 'data/stock' / f"stock_{item['code']}.csv"
        out.to_csv(path, index=False, encoding='utf-8-sig')
        log('SUCCESS', f"stock_{item['code']}", f"shape={out.shape}")
        time.sleep(0.3)
    except Exception as e:
        log('FAILED', f"stock_{item['code']}", f"Error: {e}")


[2026-05-22 00:46:11] SUCCESS stock_000625       shape=(1544, 10)


[2026-05-22 00:46:11] SUCCESS stock_601127       shape=(1543, 10)


[2026-05-22 00:46:12] SUCCESS stock_600246       shape=(1543, 10)


[2026-05-22 00:46:13] SUCCESS stock_000560       shape=(1544, 10)


[2026-05-22 00:46:14] SUCCESS stock_600519       shape=(1544, 10)


[2026-05-22 00:46:15] SUCCESS stock_000799       shape=(1544, 10)


[2026-05-22 00:46:16] SUCCESS stock_600578       shape=(1544, 10)


[2026-05-22 00:46:17] SUCCESS stock_601016       shape=(1538, 10)


[2026-05-22 00:46:17] SUCCESS stock_000063       shape=(1543, 10)


[2026-05-22 00:46:18] SUCCESS stock_300136       shape=(1544, 10)


## 1.2 市场指数

指数数据使用 `akshare.stock_zh_index_daily()` 下载沪深 300 和上证综指。沪深 300 是后续 CAPM 的市场基准；上证综指用于补充观察 A 股整体大盘走势。


In [4]:
for item in INDEXES:
    try:
        df = ak.stock_zh_index_daily(symbol=item['ak_symbol'])
        if df.empty:
            raise ValueError('No data returned')
        out = normalize_index_df(df, item)
        out = out[out['date'] >= pd.to_datetime(START_DATE_DASH)]
        path = PROJECT_ROOT / 'data/index' / f"index_{item['code']}.csv"
        out.to_csv(path, index=False, encoding='utf-8-sig')
        log('SUCCESS', f"index_{item['code']}", f"shape={out.shape}")
        time.sleep(0.3)
    except Exception as e:
        log('FAILED', f"index_{item['code']}", f"Error: {e}")


[2026-05-22 00:46:19] SUCCESS index_000300       shape=(1544, 9)


[2026-05-22 00:46:19] SUCCESS index_000001       shape=(1544, 9)


## 1.3 宏观经济指标

宏观变量保留 CPI 同比和 M2 同比。它们都是月度数据，后面清洗时再按月份和交易日数据对齐。


In [5]:

try:
    cpi = ak.macro_china_cpi_yearly().copy()
    cpi = cpi.rename(columns={'日期': 'date', '今值': 'value'})
    cpi = cpi[['date', 'value']].dropna(subset=['date'])
    cpi['date'] = pd.to_datetime(cpi['date'])
    cpi['indicator'] = 'cpi_yoy'
    cpi = cpi[cpi['date'] >= START_DATE_DASH].sort_values('date')
    cpi.to_csv(PROJECT_ROOT / 'data/macro/macro_cpi.csv', index=False, encoding='utf-8-sig')
    log('SUCCESS', 'macro_cpi', f'shape={cpi.shape}')
except Exception as e:
    log('FAILED', 'macro_cpi', f'Error: {e}')

try:
    m2 = ak.macro_china_money_supply().copy()
    m2 = m2.rename(columns={'月份': 'date', '货币和准货币(M2)-同比增长': 'value'})
    m2 = m2[['date', 'value']].dropna(subset=['date'])
    m2['date'] = pd.to_datetime(m2['date'].astype(str).str.replace('年', '-').str.replace('月份', '-01'))
    m2['indicator'] = 'm2_yoy'
    m2 = m2[m2['date'] >= START_DATE_DASH].sort_values('date')
    m2.to_csv(PROJECT_ROOT / 'data/macro/macro_m2.csv', index=False, encoding='utf-8-sig')
    log('SUCCESS', 'macro_m2', f'shape={m2.shape}')
except Exception as e:
    log('FAILED', 'macro_m2', f'Error: {e}')


[2026-05-22 00:46:23] SUCCESS macro_cpi          shape=(70, 3)
[2026-05-22 00:46:23] SUCCESS macro_m2           shape=(76, 3)


## 1.4 财务指标

财务指标来自 `akshare.stock_financial_analysis_indicator()`。这里只取作业中会用到的几类指标，并整理成 `code, year, indicator, value` 这种长表，便于后面合并和画图。


In [6]:

finance_rows = []
indicator_map = {
    '净资产收益率(%)': 'roe',
    '销售净利率(%)': 'net_profit_margin',
    '主营业务收入增长率(%)': 'revenue_growth',
}

for item in STOCKS:
    try:
        raw = ak.stock_financial_analysis_indicator(symbol=item['code'], start_year='2020')
        if raw.empty:
            raise ValueError('No financial data returned')
        raw['日期'] = pd.to_datetime(raw['日期'])
        raw['year'] = raw['日期'].dt.year
        annual = raw[raw['日期'].dt.month.eq(12)].copy()
        if annual.empty:
            annual = raw.sort_values('日期').groupby('year', as_index=False).tail(1)
        annual = annual.sort_values('year').tail(5)
        for _, row in annual.iterrows():
            for raw_col, indicator in indicator_map.items():
                if raw_col in annual.columns:
                    finance_rows.append({
                        'code': item['code'], 'name': item['name'], 'industry': item['industry'],
                        'year': int(row['year']), 'indicator': indicator,
                        'value': pd.to_numeric(row.get(raw_col), errors='coerce'),
                        'source_field': raw_col
                    })
        log('SUCCESS', f"finance_{item['code']}", f"years={annual['year'].min()}-{annual['year'].max()}, rows={len(annual)}")
        time.sleep(0.5)
    except Exception as e:
        log('FAILED', f"finance_{item['code']}", f'Error: {e}')

finance = pd.DataFrame(finance_rows)
finance.to_csv(PROJECT_ROOT / 'data/finance/finance_ratios.csv', index=False, encoding='utf-8-sig')
finance.head(12)


  0%|          | 0/7 [00:00<?, ?it/s]

[2026-05-22 00:46:24] SUCCESS finance_000625     years=2021-2025, rows=5


  0%|          | 0/7 [00:00<?, ?it/s]

[2026-05-22 00:46:26] SUCCESS finance_601127     years=2021-2025, rows=5


  0%|          | 0/7 [00:00<?, ?it/s]

...

  0%|          | 0/7 [00:00<?, ?it/s]

[2026-05-22 00:46:40] SUCCESS finance_000063     years=2021-2025, rows=5


  0%|          | 0/7 [00:00<?, ?it/s]

[2026-05-22 00:46:42] SUCCESS finance_300136     years=2021-2025, rows=5


,code,name,industry,year,indicator,value,source_field
0,000625,长安汽车,汽车,2021,roe,6.3700,净资产收益率(%)
1,000625,长安汽车,汽车,2021,net_profit_margin,3.4280,销售净利率(%)
2,000625,长安汽车,汽车,2021,revenue_growth,24.3318,主营业务收入增长率(%)
3,000625,长安汽车,汽车,2022,roe,12.4100,净资产收益率(%)
4,000625,长安汽车,汽车,2022,net_profit_margin,6.3872,销售净利率(%)
5,000625,长安汽车,汽车,2022,revenue_growth,15.3231,主营业务收入增长率(%)
6,000625,长安汽车,汽车,2023,roe,15.7600,净资产收益率(%)
7,000625,长安汽车,汽车,2023,net_profit_margin,6.2803,销售净利率(%)
8,000625,长安汽车,汽车,2023,revenue_growth,24.7787,主营业务收入增长率(%)
9,000625,长安汽车,汽车,2024,roe,9.5600,净资产收益率(%)


## 1.5 下载日志与结果检查

最后检查各类原始 CSV 是否已经生成。如果有数据源下载失败，先看 `download_log.txt` 里的报错，再单独重跑对应单元格。


In [7]:

summary = []
for folder in ['stock', 'index', 'macro', 'finance']:
    for path in sorted((PROJECT_ROOT / 'data' / folder).glob('*.csv')):
        df = pd.read_csv(path)
        summary.append({'folder': folder, 'file': path.name, 'rows': len(df), 'cols': df.shape[1]})
pd.DataFrame(summary)


,folder,file,rows,cols
0,stock,stock_000063.csv,1543,10
1,stock,stock_000560.csv,1544,10
2,stock,stock_000625.csv,1544,10
3,stock,stock_000799.csv,1544,10
4,stock,stock_300136.csv,1544,10
5,stock,stock_600246.csv,1543,10
6,stock,stock_600519.csv,1544,10
7,stock,stock_600578.csv,1544,10
8,stock,stock_601016.csv,1538,10
9,stock,stock_601127.csv,1543,10
